In [ ]:
import os
import gc
import uuid
import cv2
from componentes.autenticacion import obtener_imagen_dni_desde_wallet, obtener_usuario_desde_wallet
from componentes.autenticacion import capturar_imagen_webcam
from componentes.biometria import (verificar_biometria_tradicional, verificar_biometria_deep_learning)
from componentes.pad import evaluar_pad
from componentes.evidencias import generar_evidencia
from datetime import datetime, UTC

# ==========================================================
# FUNCIÓN AUXILIAR DE LIMPIEZA DE DATOS BIOMÉTRICOS TEMPORALES
# ==========================================================
def limpiar_biometria_temporal(*imagenes, rutas_temporales=None):
    """
    Elimina referencias en memoria a imágenes biométricas y borra
    posibles ficheros temporales de depuración si existieran.

    - No debe utilizarse para borrar evidencias lógicas.
    - Su objetivo es evitar persistencia innecesaria de biometría.
    """
    # 1. Eliminar arrays de memoria
    for img in imagenes:
        try:
            del img
        except Exception:
            pass

    # 2. Borrar ficheros temporales si se hubieran generado
    if rutas_temporales:
        for ruta in rutas_temporales:
            try:
                if ruta and os.path.exists(ruta):
                    os.remove(ruta)
            except Exception:
                pass

    # 3. Forzar liberación de memoria
    gc.collect()
# ==========================================================
# FUNCIÓN AUXILIAR DE NOTIFICACIÓN AL OPERADOR JURÍDICO
# ==========================================================
def notificar_operador_juridico(sesion_verificacion, usuario, motivo):
    """
    Simula la notificación al operador jurídico.
    En un entorno real sería email, sistema interno, etc.
    """
    nombre_usuario = (
        usuario["atributos_presentados"].get("nombre")
        if usuario else "NO_IDENTIFICADO"
    )

    notificacion = {
        "timestamp_notificacion": datetime.now(UTC).isoformat(),
        "destinatario": "operador_juridico_simulado_01",
        "canal": "sistema_interno_simulado",
        "motivo": motivo,
        "session_id": sesion_verificacion["session_id"],
        "usuario": nombre_usuario,
        "estado": "ENVIADA"
    }

    print("\n📢 NOTIFICACIÓN A OPERADOR JURÍDICO")
    print(f"Sesión: {sesion_verificacion['session_id']}")
    print(f"Usuario: {nombre_usuario}")
    print(f"Motivo: {motivo}")

    return notificacion

# ==========================================================
# FUNCIÓN AUXILIAR DE REVISIÓN HUMANA POR OPERADOR JURÍDICO
# ==========================================================
def revision_humana(usuario):
    """
    Simulación de revisión manual por operador jurídico.
    """
    print("\n👨‍⚖️ REVISIÓN MANUAL EN CURSO")

    # Puedes hacerlo automático (más simple) o interactivo
    decision_operador = "VERIFICACION_POSITIVA"

    resultado = {
        "decision_operador": decision_operador,
        "timestamp_revision": datetime.now(UTC).isoformat(),
        "operador": "operador_simulado_01",
        "observaciones": "Validación visual satisfactoria"
    }

    print("✅ Revisión manual completada:", decision_operador)

    return resultado

# ==========================================================
# INICIALIZACIÓN DEL PROCESO Y ASOCIACIÓN A LA ACTUACIÓN
# ==========================================================
# Se crea un contexto de sesión de verificación asociado a una
# actuación judicial concreta. Esto permite trazabilidad,
# auditoría y vinculación posterior de la evidencia.

sesion_verificacion = {
    "session_id": str(uuid.uuid4()),
    "id_actuacion_judicial": "EJE-2026-0001",
    "organo_judicial": "Juzgado_simulado_01",
    "timestamp_inicio": datetime.now(UTC).isoformat(),
    "estado_proceso": "INICIADO"
}

# ==========================================================
# VARIABLES DE CONTROL DEL ORQUESTADOR
# ==========================================================
usuario = None
usuario_wallet = None
img_dni = None
img_camara = None
evidencia = None
resultado_revision_humana = None
notificacion_operador = None
decision_automatica = None
motivo_decision_automatica = None

resultado_lbph = {
    "metodo": "LBPH",
    "metric_type": "lbph_confidence",
    "metric_value": None,
    "thresholds": {
        "match_fuerte": 50,
        "zona_gris": 70
    },
    "estado": "NO_EJECUTADO"
}

resultado_dl = {
    "metodo": "DeepLearning",
    "metric_type": "cosine_similarity",
    "metric_value": None,
    "thresholds": {
        "match_fuerte": 0.75,
        "zona_gris": 0.55
    },
    "estado": "NO_EJECUTADO"
}

resultado_pad = {
    "metodo": "PAD_blur",
    "laplacian_var": None,
    "estado": "NO_EJECUTADO"
}

resultados_tecnicos = {
    "biometria_tradicional": resultado_lbph,
    "biometria_deeplearning": resultado_dl,
    "pad": resultado_pad
}

decision_final = "PENDIENTE"
motivo_decision = None
incidencia = None

# Si en algún momento generas ficheros temporales, añádelos aquí
rutas_temporales = []

try:
    # ==========================================================
    # FASE 1: PRESENTACIÓN DE IDENTIDAD Y CONSENTIMIENTO
    # ==========================================================
    # El usuario presenta su identidad mediante wallet y otorga
    # consentimiento para el tratamiento puntual de biometría.

    presentacion = {
        "timestamp": datetime.now(UTC).isoformat(),
        "consentimiento_usuario": True,
        "metodo": "wallet_digital"
    }

    # El consentimiento no solo se declara: gobierna el flujo.
    if not presentacion["consentimiento_usuario"]:
        decision_final = "CANCELADO_SIN_CONSENTIMIENTO"
        motivo_decision = (
            "El usuario no ha otorgado consentimiento para "
            "el tratamiento de datos biométricos."
        )
        sesion_verificacion["estado_proceso"] = "FINALIZADO_SIN_VERIFICACION"
        raise PermissionError(motivo_decision)

    # ==========================================================
    # FASE 2: IDENTIFICACIÓN PREVIA
    # ==========================================================
    did_usuario = "did:example:8f3a9c2d"

    # ==========================================================
    # FASE 3: RECUPERACIÓN DE ATRIBUTOS DESDE LA WALLET
    # ==========================================================
    usuario_wallet = obtener_usuario_desde_wallet(did_usuario)

    # ==========================================================
    # FASE 4: OBTENCIÓN DE IMAGEN DE REFERENCIA
    # ==========================================================
    img_dni = obtener_imagen_dni_desde_wallet(did_usuario)

    # ==========================================================
    # FASE 5: CONSTRUCCIÓN DEL CONTEXTO DE USUARIO
    # ==========================================================
    usuario = {
        "did": did_usuario,
        "rol": "interviniente",
        "origen": "wallet_simulado",
        "atributos_presentados": {
            "nombre": usuario_wallet["identidad"]["nombre"],
            "apellidos": usuario_wallet["identidad"]["apellidos"],
            "dni": usuario_wallet["documento"]["numero"]
        }
    }

    # ==========================================================
    # FASE 6: CAPTURA DE IMAGEN EN TIEMPO REAL
    # ==========================================================
    img_camara = capturar_imagen_webcam()

    
    
    # =========================
    # DEBUG VISUAL
    # =========================
    cv2.imshow("Imagen Cámara", img_camara)
    cv2.imshow("Imagen DNI", img_dni)
    
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    # ==========================================================
    # FASE 7: VERIFICACIÓN BIOMÉTRICA
    # ==========================================================
    # 1) Biométrica tradicional (secundaria)
    resultado_lbph = verificar_biometria_tradicional(
        img_dni,
        img_camara
    )

    # 2) Biométrica Deep Learning (principal)
    # Este orquestador asume que la función corregida devuelve:
    # - metric_type = "cosine_similarity"
    # - mayor similitud = mejor coincidencia
    resultado_dl = verificar_biometria_deep_learning(
        img_dni,
        img_camara
    )

    # ==========================================================
    # FASE 8: EVALUACIÓN PAD
    # ==========================================================
    resultado_pad = evaluar_pad(img_camara)
    estado_pad = resultado_pad["estado"]

    # ==========================================================
    # FASE 9: AGRUPACIÓN DE RESULTADOS TÉCNICOS
    # ==========================================================
    resultados_tecnicos = {
        "biometria_tradicional": resultado_lbph,
        "biometria_deeplearning": resultado_dl,
        "pad": resultado_pad
    }

    # ==========================================================
    # FASE 10: DECISIÓN FINAL DEL ORQUESTADOR
    # ==========================================================
    # Regla jerárquica:
    # 1. El PAD tiene prioridad absoluta.
    # 2. Si PAD es válido, manda el modelo principal DL.
    # 3. Casos anómalos o ambiguos => revisión humana.

    # 10.1. Escenarios alternativos por PAD
    if estado_pad != "bonafide":
        decision_final = "REVISION_HUMANA"
        motivo_decision = (
            "El módulo PAD no ha clasificado la captura como bonafide."
        )

    else:
        # 10.2. Escenarios alternativos del biométrico principal
        estado_biometria = resultado_dl["estado"]

        if estado_biometria == "SATISFACTORIO":
            decision_final = "VERIFICACION_POSITIVA"
            motivo_decision = (
                "PAD válido y biometría principal satisfactoria."
            )

        elif estado_biometria in ["INCONCLUSO", "RIESGO", "RESULTADO_ANOMALO"]:
            decision_final = "REVISION_HUMANA"
            motivo_decision = (
                "Resultado biométrico no concluyente, de riesgo o anómalo."
            )

        else:
            decision_final = "REVISION_HUMANA"
            motivo_decision = (
                "Estado biométrico no reconocido por el orquestador."
            )

    sesion_verificacion["estado_proceso"] = "FINALIZADO"

except PermissionError as e:
    incidencia = {
        "tipo": "CONSENTIMIENTO",
        "detalle": str(e)
    }
    if decision_final == "PENDIENTE":
        decision_final = "CANCELADO_SIN_CONSENTIMIENTO"
    if motivo_decision is None:
        motivo_decision = str(e)

except (FileNotFoundError, ValueError, IOError) as e:
    # Errores de identidad, wallet, imagen de referencia, modelos, etc.
    incidencia = {
        "tipo": "ERROR_TECNICO_CONTROLADO",
        "detalle": str(e)
    }
    decision_final = "REVISION_HUMANA"
    motivo_decision = (
        "Incidencia técnica controlada durante la recuperación "
        "de identidad o biometría."
    )
    sesion_verificacion["estado_proceso"] = "FINALIZADO_CON_INCIDENCIA"

except RuntimeError as e:
    # Errores operativos como ausencia de webcam o fallo de captura
    incidencia = {
        "tipo": "ERROR_OPERATIVO",
        "detalle": str(e)
    }
    decision_final = "REVISION_HUMANA"
    motivo_decision = (
        "Incidencia operativa durante la captura o ejecución del flujo."
    )
    sesion_verificacion["estado_proceso"] = "FINALIZADO_CON_INCIDENCIA"

except Exception as e:
    # Fallback defensivo para cualquier caso no contemplado
    incidencia = {
        "tipo": "ERROR_NO_CONTROLADO",
        "detalle": str(e)
    }
    decision_final = "REVISION_HUMANA"
    motivo_decision = (
        "Se ha producido una incidencia no controlada. "
        "El caso debe escalarse a revisión humana."
    )
    sesion_verificacion["estado_proceso"] = "FINALIZADO_CON_INCIDENCIA"

finally:
    # ==========================================================
    # FASE 11: REVISIÓN HUMANA (SI APLICA)
    # ==========================================================
    
    if decision_final == "REVISION_HUMANA":

        # Guardar la decisión automática previa
        decision_automatica = decision_final
        motivo_decision_automatica = motivo_decision

        notificacion_operador = notificar_operador_juridico(
            sesion_verificacion,
            usuario,
            motivo_decision
        )

        resultado_revision_humana = revision_humana(usuario)

        # La decisión final pasa a ser la del operador
        decision_final = resultado_revision_humana["decision_operador"]
        motivo_decision = "Decisión final validada por operador jurídico"

        sesion_verificacion["estado_proceso"] = "FINALIZADO_CON_REVISION_HUMANA"
    # ==========================================================
    # FASE 12: GENERACIÓN DE EVIDENCIA ELECTRÓNICA
    # ==========================================================
    # La evidencia debe incluir:
    # - identidad presentada
    # - sesión / actuación judicial
    # - resultados técnicos
    # - decisión final
    # - motivo e incidencias si las hubiera

    evidencia = generar_evidencia(
        usuario=usuario,
        sesion_verificacion=sesion_verificacion,
        resultados_tecnicos=resultados_tecnicos,
        decision_final=decision_final,
        motivo_decision=motivo_decision,
        incidencia=incidencia,
        revision_humana=resultado_revision_humana,
        notificacion_operador=notificacion_operador,
        decision_automatica=decision_automatica,
        motivo_decision_automatica=motivo_decision_automatica
    )


    # ==========================================================
    # FASE 13: ELIMINACIÓN DE DATOS BIOMÉTRICOS TEMPORALES
    # ==========================================================
    limpiar_biometria_temporal(
        img_dni,
        img_camara,
        rutas_temporales=rutas_temporales
    )

    # ==========================================================
    # VISUALIZACIÓN DE RESULTADOS (SOLO PRUEBAS)
    # ==========================================================
    print("📄 Evidencia generada:")
    print(evidencia)

    print("\n✅ Decisión final:")
    print(decision_final)

    if motivo_decision:
        print("\n🧭 Motivo de la decisión:")
        print(motivo_decision)

    if incidencia:
        print("\n⚠️ Incidencia registrada:")
        print(incidencia)


Cámara 0: OK
Cámara 1: OK
Cámara 2: NO FUNCIONA
Cámara 3: NO FUNCIONA
📷 Pulsa ENTER para capturar tu cara...
